# Vocabulary Space Analysis

Analyze projections into vocabulary space: logit neighbors, coverage,
prediction diversity, token trajectories, and summary.

## Why This Matters

This module provides tools for analyzing model internals. Understanding the internal mechanisms of transformer models is the core goal of mechanistic interpretability research — enabling us to move from treating models as black boxes to understanding the algorithms they implement.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.vocabulary_space_analysis import (
    logit_space_neighbors, vocabulary_coverage,
    prediction_diversity_across_positions, token_logit_trajectory,
    vocabulary_space_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Logit Space Neighbors

What tokens are nearest in logit space at each layer?

In [ ]:
for layer in range(4):
    result = logit_space_neighbors(model, tokens, position=-1, layer=layer, top_k=5)
    top3 = ', '.join(f'tok{n["token_id"]}({n["probability"]:.3f})' for n in result['neighbors'][:3])
    print(f"  Layer {layer}: top={result['top_token']} ({result['top_prob']:.4f}) | {top3}")

## Vocabulary Coverage

How many tokens are assigned non-negligible probability?

In [ ]:
result = vocabulary_coverage(model, tokens, layer=-1)
print(f"Mean coverage: {result['mean_coverage']:.1f} tokens\n")
for p in result['per_position']:
    bar = '█' * int(min(p['n_tokens_above_threshold'], 30))
    print(f"  Pos {p['position']} (tok {p['token_id']:3d}): "
          f"n={p['n_tokens_above_threshold']}, H={p['entropy']:.3f} {bar}")

## Prediction Diversity

Are different positions making different predictions?

In [ ]:
for layer in range(4):
    result = prediction_diversity_across_positions(model, tokens, layer=layer)
    flag = ' [DIVERSE]' if result['is_diverse'] else ''
    preds = ', '.join(str(t) for t in result['top_tokens'])
    print(f"  Layer {layer}: unique={result['n_unique_predictions']}, "
          f"sim={result['mean_logit_similarity']:.4f}{flag}")
    print(f"    predictions: [{preds}]")

## Token Logit Trajectory

Track a specific token's logit across all layers.

In [ ]:
for target in [30, 45, 60]:
    result = token_logit_trajectory(model, tokens, target_token=target)
    print(f"  Token {target}: final_rank={result['final_rank']}")
    for p in result['per_layer']:
        print(f"    Layer {p['layer']}: logit={p['target_logit']:+.4f}, "
              f"prob={p['target_prob']:.4f}, rank={p['target_rank']}")

## Vocabulary Space Summary

Cross-layer overview of vocabulary space projections.

In [ ]:
result = vocabulary_space_summary(model, tokens)
print(f"Sharpening: {result['sharpening']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['mean_max_prob'] * 30)
    print(f"  Layer {p['layer']}: H={p['mean_entropy']:.3f}, "
          f"max_p={p['mean_max_prob']:.4f}, "
          f"unique={p['n_unique_predictions']} {bar}")